# American Option Pricing — NVIDIA GPU (Google Colab)

**Research extension by Xiaohu Zhang**, based on the OpenCL implementation by Leon Xing Li.

**Methods:**
- Monte Carlo European option (NumPy CPU vs CUDA GPU)
- PSO American option (NumPy CPU vs CUDA GPU: hybrid / scalar / fusion)
- Longstaff-Schwartz LSMC (NumPy CPU vs CUDA GPU)

> **Runtime:** Make sure you have selected **GPU** under *Runtime → Change runtime type → T4 GPU*

## 1. Setup

In [ ]:
# Install PyCUDA (only needed once per Colab session)
!pip install pycuda --quiet

In [ ]:
import subprocess, os

# Clone the CUDA port repo if not already present
repo_dir = '/content/CUDAPSOAMERICAOPTION'
repo_url = 'https://github.com/xiaohuzhang19/CUDAPSOAMERICAOPTION'
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)

os.chdir(os.path.join(repo_dir, 'src'))
print('Working directory:', os.getcwd())

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
from models.utils import checkCUDA, CUDAEnv
checkCUDA()
print('Device:', CUDAEnv.deviceName)

## 2. Simulation Parameters

In [ ]:
# Option parameters
S0      = 100.0    # spot price
K       = 100.0    # strike price
r       = 0.03     # risk-free rate
sigma   = 0.30     # volatility
T       = 1.0      # maturity (years)
opttype = 'P'      # 'P' = Put, 'C' = Call

# Simulation parameters
nPath   = 65536    # Monte Carlo paths
nPeriod = 50       # time steps
nFish   = 500      # PSO particles

print(f'Paths={nPath:,}  Periods={nPeriod}  PSO particles={nFish}')

## 3. Monte Carlo — European Option

In [ ]:
from models.mc import hybridMonteCarlo

mc = hybridMonteCarlo(S0, r, sigma, T, nPath, nPeriod, K, opttype, nFish)

# CPU baseline
price_np, t_np = mc.getEuroOption_np()

# GPU – one-shot optimized (WINNER)
price_gpu, t_gpu = mc.getEuroOption_cl_optimized()

print(f'\nSpeedup: {t_np/t_gpu:.1f}x')

## 4. PSO — American Option

In [ ]:
from models.pso import PSO_Numpy, PSO_CUDA_hybrid, PSO_CUDA_scalar, PSO_CUDA_scalar_fusion

iterMax = 30

# CPU
pso_np = PSO_Numpy(mc, nFish, iterMax=iterMax)
price_pso_np, t_pso_np, *_ = pso_np.solvePsoAmerOption_np()

# GPU – hybrid
pso_hyb = PSO_CUDA_hybrid(mc, nFish, iterMax=iterMax)
price_hyb, t_hyb, *_ = pso_hyb.solvePsoAmerOption_cl()

# GPU – scalar (backward)
pso_sc = PSO_CUDA_scalar(mc, nFish, direction='backward', iterMax=iterMax)
price_sc, t_sc, *_ = pso_sc.solvePsoAmerOption_cl()

# GPU – fused
pso_fus = PSO_CUDA_scalar_fusion(mc, nFish, iterMax=iterMax)
price_fus, t_fus, *_ = pso_fus.solvePsoAmerOption_cl()

print(f'\nSpeedup (hybrid) : {t_pso_np/t_hyb:.1f}x')
print(f'Speedup (scalar) : {t_pso_np/t_sc:.1f}x')
print(f'Speedup (fusion) : {t_pso_np/t_fus:.1f}x')

## 5. Longstaff-Schwartz LSMC — American Option

In [ ]:
from models.longstaff import LSMC_Numpy, LSMC_CUDA

# CPU
ls_np  = LSMC_Numpy(mc, inverseType='benchmark_pinv')
price_ls_np, t_ls_np = ls_np.longstaff_schwartz_itm_path_fast()

# GPU – GJ (default)
ls_gpu = LSMC_CUDA(mc, preCalc=None, inverseType='GJ')
price_ls_gpu, t_ls_gpu = ls_gpu.longstaff_schwartz_itm_path_fast_hybrid()

# GPU – optimized
ls_gpu_opt = LSMC_CUDA(mc, preCalc='optimized')
price_ls_gpu_opt, t_ls_gpu_opt = ls_gpu_opt.longstaff_schwartz_itm_path_fast_hybrid()

print(f'\nSpeedup (LSMC-GJ)  : {t_ls_np/t_ls_gpu:.1f}x')
print(f'Speedup (LSMC-opt) : {t_ls_np/t_ls_gpu_opt:.1f}x')

## 6. Summary

In [ ]:
import pandas as pd

results = {
    'Method'  : ['MC-NumPy', 'MC-CUDA', 'PSO-NumPy', 'PSO-CUDA-hybrid', 'PSO-CUDA-scalar', 'PSO-CUDA-fusion', 'LSMC-NumPy', 'LSMC-CUDA-GJ', 'LSMC-CUDA-opt'],
    'Price'   : [price_np, price_gpu, price_pso_np, price_hyb, price_sc, price_fus, price_ls_np, price_ls_gpu, price_ls_gpu_opt],
    'Time(ms)': [t_np, t_gpu, t_pso_np, t_hyb, t_sc, t_fus, t_ls_np, t_ls_gpu, t_ls_gpu_opt],
}

df = pd.DataFrame(results)
df['Price'] = df['Price'].map('{:.4f}'.format)
df['Time(ms)'] = df['Time(ms)'].map('{:.2f}'.format)
print(df.to_string(index=False))

In [ ]:
# Free Z_d on device
mc.cleanUp()

---
## 7. Empirical Study — S&P 500 September 2008 (TARP Rejection Day)

**Context:** On September 29, 2008, the House voted down the TARP bailout bill.
The S&P 500 fell ~8.8% in a single session (1106.42 close). This study prices
near-expiry American Put options at three moneyness levels using the implied
volatilities observed that day.

| Case  | S0      | K        | r      | σ        | T (days) | Market Price |
|-------|---------|----------|--------|----------|----------|--------------|
| ITM   | 1106.42 | 1174.136 | 0.0094 | 0.282437 | 30       | 78.98        |
| ATM   | 1106.42 | 1106.000 | 0.0094 | 0.275000 | 30       | 34.01        |
| vOTM  | 1106.42 |  987.000 | 0.0094 | 0.410000 | 30       | 10.74        |

In [ ]:
# ── Sep 2008 empirical portfolio ────────────────────────────────────────────
# Each tuple: (label, S0, K, r, sigma, T_days, opttype, market_price, nPath, nPeriod, nFish)
PORTFOLIO = [
    ('ITM  Put (d=-75)', 1106.42, 1174.136, 0.0094, 0.282437, 30, 'P', 78.9833, 10000, 200, 256),
    ('ATM  Put (d=-50)', 1106.42, 1106.000, 0.0094, 0.275000, 30, 'P', 34.0100, 10000, 200, 256),
    ('vOTM Put (d=-15)', 1106.42,  987.000, 0.0094, 0.410000, 30, 'P', 10.7400, 10000, 200, 256),
]
print(f'Cases to run: {len(PORTFOLIO)}')
for label, S0, K, r, sigma, T_days, opttype, mkt, nPath, nPeriod, nFish in PORTFOLIO:
    print(f'  {label}: S0={S0}, K={K}, σ={sigma}, T={T_days}d, market={mkt}')

In [ ]:
import pandas as pd
from models.mc import hybridMonteCarlo, BlackScholes
from models.pso import PSO_Numpy, PSO_CUDA_scalar_fusion
from models.longstaff import LSMC_Numpy, LSMC_CUDA

iterMax = 30
all_rows = []

for label, S0, K, r, sigma, T_days, opttype, mkt, nPath, nPeriod, nFish in PORTFOLIO:
    T = T_days / 365.0
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'  S0={S0}  K={K}  r={r}  σ={sigma}  T={T_days}d  market={mkt}')
    print(f'{"="*60}')

    # Build MC object (generates paths)
    mc_emp = hybridMonteCarlo(S0, r, sigma, T, nPath, nPeriod, K, opttype, nFish)

    # ── Black-Scholes reference (European)
    bs_price = BlackScholes(S0, K, r, sigma, T, opttype)
    print(f'  Black-Scholes (European ref): {bs_price:.4f}')

    # ── PSO CPU baseline
    pso_cpu = PSO_Numpy(mc_emp, nFish, iterMax=iterMax)
    price_pso_cpu, t_pso_cpu, *_ = pso_cpu.solvePsoAmerOption_np()
    print(f'  PSO CPU  : {price_pso_cpu:.4f}  ({t_pso_cpu:.1f} ms)')

    # ── PSO GPU fusion
    pso_gpu = PSO_CUDA_scalar_fusion(mc_emp, nFish, iterMax=iterMax)
    price_pso_gpu, t_pso_gpu, *_ = pso_gpu.solvePsoAmerOption_cl()
    speedup_pso = t_pso_cpu / t_pso_gpu
    print(f'  PSO GPU  : {price_pso_gpu:.4f}  ({t_pso_gpu:.1f} ms)  speedup={speedup_pso:.1f}x')

    # ── LSMC CPU baseline
    ls_cpu = LSMC_Numpy(mc_emp, inverseType='benchmark_pinv')
    price_ls_cpu, t_ls_cpu = ls_cpu.longstaff_schwartz_itm_path_fast()
    print(f'  LSMC CPU : {price_ls_cpu:.4f}  ({t_ls_cpu:.1f} ms)')

    # ── LSMC GPU optimized
    ls_gpu = LSMC_CUDA(mc_emp, preCalc='optimized')
    price_ls_gpu, t_ls_gpu = ls_gpu.longstaff_schwartz_itm_path_fast_hybrid()
    speedup_ls = t_ls_cpu / t_ls_gpu
    print(f'  LSMC GPU : {price_ls_gpu:.4f}  ({t_ls_gpu:.1f} ms)  speedup={speedup_ls:.1f}x')

    # ── collect results
    all_rows.append({
        'Case'          : label,
        'Market'        : f'{mkt:.4f}',
        'BS (European)' : f'{bs_price:.4f}',
        'PSO CPU'       : f'{price_pso_cpu:.4f}',
        'PSO GPU'       : f'{price_pso_gpu:.4f}',
        'PSO Speedup'   : f'{speedup_pso:.1f}x',
        'LSMC CPU'      : f'{price_ls_cpu:.4f}',
        'LSMC GPU'      : f'{price_ls_gpu:.4f}',
        'LSMC Speedup'  : f'{speedup_ls:.1f}x',
    })

    mc_emp.cleanUp()

print('\nAll cases done.')

In [ ]:
# ── Final summary table ──────────────────────────────────────────────────────
import pandas as pd

df_emp = pd.DataFrame(all_rows)
print('\nS&P 500 Sep 29, 2008 — American Put Pricing Summary')
print('Parameters: nPath=10,000  nPeriod=200  nFish=256  iterMax=30')
print()
print(df_emp.to_string(index=False))